In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks
import os

from pre_processing import preprocess_image


In [ ]:
IMAGE_PATH = "./dataset/word2.png"
OUTPUT_DIR = "segmented_letters"
IMG_SIZE = 128
EXPECTED_LETTERS = 2  # number of letters in the word

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
img = cv2.imread(IMAGE_PATH)

if img is None:
    raise ValueError("Image not found")

clean = img_processing(img)

plt.figure(figsize=(8,4))
plt.imshow(clean, cmap="gray")
plt.title("After Pre-processing")
plt.axis("off")
plt.show()


In [ ]:
thresh = cv2.adaptiveThreshold(
    clean, 255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY_INV,
    15, 4
)

plt.figure(figsize=(8,4))
plt.imshow(thresh, cmap="gray")
plt.title("After Adaptive Threshold")
plt.axis("off")
plt.show()


In [ ]:
kernel_h = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 14))
horizontal_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_h)
clean = cv2.subtract(thresh, horizontal_lines)

plt.figure(figsize=(8,4))
plt.imshow(clean, cmap="gray")
plt.title("After Removing Horizontal Noise")
plt.axis("off")
plt.show()


In [ ]:
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
clean = cv2.erode(clean, kernel, iterations=1)

plt.figure(figsize=(8,4))
plt.imshow(clean, cmap="gray")
plt.title("After Erosion")
plt.axis("off")
plt.show()


In [ ]:
projection = np.sum(clean, axis=0)
projection = projection / np.max(projection)

smooth = savgol_filter(projection, 51, 3)

plt.figure(figsize=(8,4))
plt.plot(smooth)
plt.title("Smoothed Vertical Projection")
plt.xlabel("X (columns)")
plt.ylabel("Ink Density")
plt.show()


In [ ]:
peaks, _ = find_peaks(smooth, distance=150)

# Limit peaks to expected number of letters
if len(peaks) > EXPECTED_LETTERS:
    peak_values = smooth[peaks]
    strongest = np.argsort(peak_values)[-EXPECTED_LETTERS:]
    peaks = np.sort(peaks[strongest])

print("Final peaks:", peaks)


In [ ]:
cuts = []

for i in range(len(peaks) - 1):
    l, r = peaks[i], peaks[i + 1]
    cut = np.argmin(smooth[l:r]) + l
    cuts.append(cut)

print("Cut positions:", cuts)


In [ ]:
letters = []
prev = 0

for c in cuts:
    letters.append(img[:, prev:c])
    prev = c

letters.append(img[:, prev:])

print("Number of letters:", len(letters))


In [ ]:
plt.figure(figsize=(10,3))

for i, letter in enumerate(letters):
    resized = cv2.resize(letter, (IMG_SIZE, IMG_SIZE))
    cv2.imwrite(f"{OUTPUT_DIR}/letter_{i}.png", resized)

    plt.subplot(1, len(letters), i + 1)
    plt.imshow(resized, cmap="gray")
    plt.axis("off")
    plt.title(f"L{i}")

plt.show()

print(f"Letters saved in: {OUTPUT_DIR}")
